In [3]:
import os
import ee
from Software_Crawler.download import initialize_ee
from dotenv import load_dotenv

load_dotenv()  # Carica le variabili d'ambiente dal file .env

def get_histogram_data(
    lat: float, 
    lon: float,
    project: str | None = None,
    service_account: str | None = None,
    service_account_key: str | None = None
) -> dict:
    """
    Recupera i dati dell'istogramma per una data latitudine e longitudine,
    calcolando la percentuale di ogni classe colturale presente nel buffer.

    Args:
        lat (float): Latitudine del punto centrale.
        lon (float): Longitudine del punto centrale.
        project (str | None): ID del progetto Google Cloud.
        service_account (str | None): Email del Service Account.
        service_account_key (str | None): Percorso della chiave JSON.

    Returns:
        dict: Un dizionario contenente latitudine, longitudine e le percentuali delle classi.
    """
    
    # 1. Inizializzazione
    project = project or os.getenv("GOOGLE_CLOUD_PROJECT")
    service_account = service_account or os.getenv("GEE_SERVICE_ACCOUNT")
    service_account_key = service_account_key or os.getenv("GEE_SERVICE_ACCOUNT_KEY")
    
    initialize_ee(project, service_account, service_account_key)
    
    # 2. Definizione dell'area di interesse (Buffer di 1000m)
    roi = ee.Geometry.Point([lon, lat]).buffer(1000)
    
    # 3. Caricamento del dataset NASS CDL e selezione della banda
    nass_dataset = ee.ImageCollection("USDA/NASS/CDL").filterDate('2023-01-01', '2023-12-31').first()
    cropland = nass_dataset.select('cropland')
    
    # 4. Calcolo dell'istogramma delle frequenze su Google Earth Engine
    stats = cropland.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=roi,
        scale=30, # Risoluzione nativa del NASS
        maxPixels=1e9
    )
    
    # 5. Trasferimento in locale del dizionario
    histogram_dict = stats.getInfo()
    
    # Estrazione sicura del conteggio dei pixel
    pixel_counts = histogram_dict.get('cropland')
    
    # Se l'area non ha dati (es. oceano o fuori mappa), restituisci un avviso
    if not pixel_counts:
        return {
            "lat": lat, 
            "lon": lon, 
            "error": "Nessun dato NASS disponibile in quest'area."
        }
        
    # 6. Calcolo delle percentuali
    total_pixels = sum(pixel_counts.values())
    percentages_dict = {}
    
    for class_val, count in pixel_counts.items():
        # Calcola la percentuale
        perc = (count / total_pixels) * 100
        # Salva la classe come chiave (es. 'Classe_1') e arrotonda la percentuale
        percentages_dict[f"Classe_{class_val}"] = round(perc, 2)
        
    # 7. Ritorno dei risultati finali
    return {
        "lat": lat,
        "lon": lon,
        "class_percentages": percentages_dict
    }

In [6]:
get_histogram_data(33.93487935325868,-90.76444348595372)  # Esempio di chiamata alla funzione

 Using local user credentials for GEE initialization...


{'lat': 33.93487935325868,
 'lon': -90.76444348595372,
 'class_percentages': {'Classe_1': 0.15,
  'Classe_111': 0.03,
  'Classe_121': 4.81,
  'Classe_122': 6.27,
  'Classe_123': 4.98,
  'Classe_124': 1.03,
  'Classe_152': 0.03,
  'Classe_176': 0.03,
  'Classe_190': 0.63,
  'Classe_195': 0.03,
  'Classe_2': 0.45,
  'Classe_3': 5.02,
  'Classe_5': 76.18,
  'Classe_74': 0.38}}

In [7]:
import pandas as pd

state_info = pd.read_csv("fao_gaul_usa_grid_with_states.csv")
crop_data = pd.read_csv("agricultural_density_results.csv")

merged_data = pd.merge(crop_data, state_info, on=['lat', 'lon'], how='left')

print(merged_data.head())  

         lat         lon  useful_crop_percentage  useless_non_crop_percentage  \
0  47.934879  -96.764443                   95.08                         4.92   
1  36.434879 -120.264443                   94.07                         5.93   
2  35.934879  -90.264443                   93.22                         6.78   
3  38.434879 -100.764443                   92.65                         7.35   
4  40.934879  -88.264443                   92.45                         7.55   

        State  
0   Minnesota  
1  California  
2    Arkansas  
3      Kansas  
4    Illinois  


In [8]:
merged_data = merged_data[0:100]
print(merged_data)  

          lat         lon  useful_crop_percentage  \
0   47.934879  -96.764443                   95.08   
1   36.434879 -120.264443                   94.07   
2   35.934879  -90.264443                   93.22   
3   38.434879 -100.764443                   92.65   
4   40.934879  -88.264443                   92.45   
..        ...         ...                     ...   
95  39.434879 -101.264443                   81.71   
96  48.934879  -97.764443                   81.71   
97  36.434879 -100.764443                   81.65   
98  42.934879  -93.264443                   81.59   
99  41.434879  -95.264443                   81.59   

    useless_non_crop_percentage         State  
0                          4.92     Minnesota  
1                          5.93    California  
2                          6.78      Arkansas  
3                          7.35        Kansas  
4                          7.55      Illinois  
..                          ...           ...  
95                        1

In [10]:
merged_data.to_csv( "Final_points.csv", index=False)